In [1]:
import pandas as pd
import numpy as np

TARGETS = ["theta"]

In [2]:
results_1l = pd.read_excel("resultados-1l-v2.xlsx")
# results_2l = pd.read_excel("resultados-2l.xlsx")
# results_3l = pd.read_excel("resultados-3l.xlsx")

# results = pd.concat(
    # [results_1l, results_2l, results_3l],
    # ignore_index=True
# )
results = results_1l


In [3]:
results

,model,Neurons,Ld,Lp,reg,seed,R2_SuperZZ1_theta,MSE_SuperZZ1_theta,R2_SuperZZ2_theta,MSE_SuperZZ2_theta,...,R2_ZZx2_theta,MSE_ZZx2_theta,R2_ZZxReto_theta,MSE_ZZxReto_theta,R2_ZZy1_theta,MSE_ZZy1_theta,R2_ZZy2_theta,MSE_ZZy2_theta,R2_semiCirc_theta,MSE_semiCirc_theta
0,model_arch1_r0.01_Ld0.3_Lp0.7_seed9701,[1],0.3,0.7,0.01,9701,0.898440,0.556300,-3.128319,0.450412,...,-4.204462,0.283367,0.178841,0.534043,-20.187454,0.106221,0.191458,0.539646,-16.587880,0.089059
1,model_arch1_r0.01_Ld0.3_Lp0.7_seed2897,[1],0.3,0.7,0.01,2897,0.897453,0.555493,-3.121104,0.450143,...,-4.198640,0.283437,0.180440,0.533163,-20.065570,0.107024,0.188088,0.539309,-16.481360,0.091414
2,model_arch1_r0.01_Ld0.3_Lp0.7_seed227,[1],0.3,0.7,0.01,227,0.903075,0.573211,-3.347734,0.453118,...,-4.335375,0.294000,0.237095,0.556651,-21.399914,0.107420,0.192361,0.545544,-19.046517,0.040695
3,model_arch1_r0.01_Ld0.3_Lp0.7_seed4497,[1],0.3,0.7,0.01,4497,0.906168,0.576210,-3.339160,0.454690,...,-4.304779,0.298246,0.251270,0.564361,-22.137176,0.102061,0.234918,0.547230,-20.375677,0.010958
4,model_arch1_r0.01_Ld0.3_Lp0.7_seed9477,[1],0.3,0.7,0.01,9477,0.905783,0.575160,-3.343618,0.454241,...,-4.311913,0.296934,0.245506,0.561831,-21.950896,0.103133,0.223521,0.546744,-19.998943,0.019333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2995,model_arch100_r0.9_Ld0.5_Lp0.5_seed713,[100],0.5,0.5,0.90,713,0.793190,0.675877,-1.636830,0.556385,...,-1.474522,0.492999,-0.798015,0.435133,-29.886095,-0.059848,0.764033,0.637633,-15.478044,0.064755
2996,model_arch100_r0.9_Ld0.5_Lp0.5_seed4130,[100],0.5,0.5,0.90,4130,0.738423,0.686034,-1.093733,0.574460,...,-2.143928,0.487990,-0.791169,0.401163,-32.196296,-0.090892,0.867657,0.655848,-14.170080,0.119757
2997,model_arch100_r0.9_Ld0.5_Lp0.5_seed2226,[100],0.5,0.5,0.90,2226,0.587846,0.701842,-0.562116,0.563798,...,-1.092246,0.501766,-2.668068,0.268649,-30.210955,-0.151773,0.940989,0.655289,-11.063085,0.159870
2998,model_arch100_r0.9_Ld0.5_Lp0.5_seed7324,[100],0.5,0.5,0.90,7324,0.762876,0.696402,-1.685119,0.568703,...,-1.146349,0.505281,-1.188602,0.404960,-28.363542,-0.038550,0.843000,0.652965,-14.144071,0.077326


In [4]:
# 🔹 categorização dos sets (baseada nos comentários originais)

SETS_CATEGORY = {
    "SuperZZ1":  "Train",
    "SuperZZ2":  "Val",
    "ZZx1":      "Test",
    "ZZx2":     "Test",
    "ZZy1":     "Test",
    "ZZy2":     "Test",
    "LSG-1":    "Test",
    "LSG-2":    "Test",
    "ZZx1-inv": "Test",
    "ZZxReto":  "Test",
    "ZZx2-inv": "Test",
    "semiCirc": "Test",
}

def col_name(s, target):
    # sanitiza "-" pra "_" pra bater com o nome real da coluna, se for o caso
    return f"R2_{s.replace('-', '_')}_{target}"

best_models_tables = {}
N = 5  # top modelos

w_val = 0.33
w_train = 0.33
w_test = 0.33

for target in TARGETS:

    # 🔹 sets de Train, Val e Test
    train_sets = [s for s, cat in SETS_CATEGORY.items() if cat == "Train"]
    val_sets   = [s for s, cat in SETS_CATEGORY.items() if cat == "Val"]
    test_sets  = [s for s, cat in SETS_CATEGORY.items() if cat == "Test"]

    train_cols = [col_name(s, target) for s in train_sets]
    val_cols   = [col_name(s, target) for s in val_sets]
    test_cols  = [col_name(s, target) for s in test_sets]

    # 🔹 garantir que só usamos colunas existentes
    train_cols = [c for c in train_cols if c in results.columns]
    val_cols   = [c for c in val_cols if c in results.columns]
    test_cols  = [c for c in test_cols if c in results.columns]

    r2_all_cols = train_cols + val_cols + test_cols

    if not r2_all_cols:
        print(f"⚠️ Nenhuma coluna Train/Val/Test encontrada para target={target}, pulando.")
        continue

    df = results.copy()

    # 🔹 remover linhas onde QUALQUER R2 (Train/Val/Test) < 0
    # df = df[(df[r2_all_cols] >= 0).all(axis=1)]

    # =========================
    # 🔹 MÉDIAS POR GRUPO
    # =========================
    df["R2_train_mean"] = df[train_cols].mean(axis=1) if train_cols else np.nan
    df["R2_val_mean"]   = df[val_cols].mean(axis=1) if val_cols else np.nan
    df["R2_test_mean"]  = df[test_cols].mean(axis=1) if test_cols else np.nan

    # =========================
    # 🔹 SCORE
    # =========================
    df["R2_std"] = df[r2_all_cols].std(axis=1)

    df["Score"] = (
        w_train * df["R2_train_mean"] +
        w_val   * df["R2_val_mean"] +
        w_test  * df["R2_test_mean"]
        - 0.1 * df["R2_std"]   # penaliza inconsistência
    )

    # =========================
    # 🔹 ORDENAÇÃO
    # =========================
    df_sorted = df.sort_values(by="Score", ascending=False)
    best_models_tables[target] = df_sorted

    # =========================
    # 🔹 TOP N RESUMO
    # =========================
    print(f"\n🏆 TOP {N} MODELOS - {target}")
    display(df_sorted[
        ["model", "Neurons", "R2_train_mean", "R2_val_mean", "R2_test_mean", "Score"]
    ].head(N))

    top_df = df_sorted.head(N).copy()

    final_cols = ["model", "Neurons"] + r2_all_cols + [
        "R2_train_mean", "R2_val_mean", "R2_test_mean", "Score"
    ]
    final_table = top_df[final_cols]

    print(f"\n📊 MÉTRICAS COMPLETAS - TOP {N} ({target})")
    display(final_table)


🏆 TOP 5 MODELOS - theta


,model,Neurons,R2_train_mean,R2_val_mean,R2_test_mean,Score
2151,model_arch16_r0.01_Ld0.5_Lp0.5_seed4130,[16],0.960271,-0.228083,-1.708803,-0.546121
1798,model_arch90_r0.9_Ld0.7_Lp0.3_seed4497,[90],0.808780,-0.071163,-3.800505,-1.374547
1451,model_arch73_r0.01_Ld0.7_Lp0.3_seed2897,[73],0.865129,-0.529590,-3.489475,-1.395208
2934,model_arch94_r0.01_Ld0.5_Lp0.5_seed2700,[94],0.687444,-0.071439,-3.881375,-1.442091
2775,model_arch78_r0.9_Ld0.5_Lp0.5_seed713,[78],0.784300,-0.949642,-3.333454,-1.493190



📊 MÉTRICAS COMPLETAS - TOP 5 (theta)


,model,Neurons,R2_SuperZZ1_theta,R2_SuperZZ2_theta,R2_ZZx1_theta,R2_ZZx2_theta,R2_ZZy1_theta,R2_ZZy2_theta,R2_LSG_1_theta,R2_LSG_2_theta,R2_ZZx1_inv_theta,R2_ZZxReto_theta,R2_semiCirc_theta,R2_train_mean,R2_val_mean,R2_test_mean,Score
2151,model_arch16_r0.01_Ld0.5_Lp0.5_seed4130,[16],0.960271,-0.228083,-3.701125,0.732972,-0.750698,0.798129,-2.245887,-2.006893,0.738242,-5.810393,-3.133573,0.960271,-0.228083,-1.708803,-0.546121
1798,model_arch90_r0.9_Ld0.7_Lp0.3_seed4497,[90],0.808780,-0.071163,-7.765532,0.281421,-3.474973,0.454798,-6.591449,-4.867659,0.638914,-8.714452,-4.165611,0.808780,-0.071163,-3.800505,-1.374547
1451,model_arch73_r0.01_Ld0.7_Lp0.3_seed2897,[73],0.865129,-0.529590,-6.293465,0.700974,-8.952923,0.640090,-5.510612,-3.232274,0.764298,-6.332850,-3.188510,0.865129,-0.529590,-3.489475,-1.395208
2934,model_arch94_r0.01_Ld0.5_Lp0.5_seed2700,[94],0.687444,-0.071439,-7.998835,0.753479,-5.656035,0.688975,-5.842587,-4.066678,0.507380,-7.846649,-5.471429,0.687444,-0.071439,-3.881375,-1.442091
2775,model_arch78_r0.9_Ld0.5_Lp0.5_seed713,[78],0.784300,-0.949642,-6.530970,0.462752,-6.038564,0.904207,-3.900512,-2.003572,0.681941,-5.400595,-8.175771,0.784300,-0.949642,-3.333454,-1.493190


In [5]:
final_table.to_excel("BestModels-1l.xlsx")